In [1]:
import gwsus as sus
import gwsus.utils as utils
import gwsus.materials as materials
import gwsus.shapes3d as shapes
from gwsus import info
from gwsus import anime

<IPython.core.display.Javascript object>

In [2]:
import sympy as sp

In [3]:
def is_symmetric(A):
    return all(
        A[i, j] == A[j, i]
        for i in range(A.rows)
        for j in range(i)
    )

# Specification of bodies and wires

In [4]:
# Ground specification
pos_GND = { 'for_F1' : utils.center( 0 ) }
GND_data = sus.GroundData( pos = pos_GND )

In [5]:
# GAS filter specification

# mass = 100.21
# f0 = 0.250
# saturation = -1e-3
# loss_angle = 1e-4
# moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 )
# pos_F1 = { 'top':utils.center( 1e-3 ) , 'bottom' : utils.center( -1e-3 ) }


mass = 91.31        # kg
f0 = 0.250          # Hz
saturation = -1e-3  # dimensionless
loss_angle = 1e-4   # dimensionless

moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 ) # kg⋅m²


# moi = utils.matrix_3x3( [ [ 3.8399, 0.61  , 0      ],
#                           [ 0.61  , 3.8399, 0      ],
#                           [ 0     , 0     , 6.4676 ] ] ) # kg⋅m²

# pos_F1 = { 'for_GND':utils.center( -13.5e-3 ),
#            'for_CB' : utils.center( -24.6e-3 ) } # args in m

pos_F1 = { 'for_GND':utils.center( 1e-3 ),
           'for_CB' : utils.center( -1e-3 ) } # args in m

F1_data = sus.GASFilterData( mass = mass,
                                    moi = moi,
                                    pos = pos_F1,
                                    resonance_frequency = f0,
                                    saturation = saturation,
                                    loss_angle = loss_angle,
                                    shape = shapes.GASFilterShape ) 

In [6]:
type( F1_data.loss_angle )

symengine.lib.symengine_wrapper.Rational

In [7]:
F1_data.mass

1606342507715625/17592186044416

In [8]:
# Mass specification
mass = 2
side = utils.con( 30e-3 )
moi = utils.con( 1 / 6 ) * mass * side**2 * utils.matrix_3x3_diagonal(1,1,1)
pos_CB = { 'for_F1' : utils.center( 3e-3 ) } 
CB_data = sus.BodyData( mass = mass ,
                        moi = moi ,
                        pos = pos_CB ,
                        shape = shapes.CubeShape )

In [9]:
print( type( CB_data.moi ) )
CB_data.moi

<class 'symengine.lib.symengine_wrapper.MutableDenseMatrix'>


[7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0, 0]
[0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0]
[0, 0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616]

In [10]:
# number = 1
# material = materials.maraging_steel
# diameter = 3e-3
# length = utils.con( 26.5e-3 + 1280e-3 + 26.5e-3 )
# wireF1_data = utils.WireData( number = number , material = material , diameter = diameter , length = length )

number = 1
material = materials.maraging_steel
diameter = ( 3e-3 , 4.5e-3 , 3e-3 )      # m
length = ( 26.5e-3 , 1280e-3 , 26.5e-3 ) # m

wireF1_data = sus.WireData( number = number,
                        material = material,
                        diameter = diameter,
                        length = length )

In [11]:
number = 1
material = materials.maraging_steel
diameter = 2.5e-3 
length = 26.5e-3 + 513.4e-3 + 26.5e-3
wireCB_data = sus.WireData( number = number , material = material , diameter = diameter , length = length )

# Declare bodies and wires and connect them

In [12]:
# Declare suspension
TS = sus.Suspension('TS')

In [13]:
# Declare body instances
GND = sus.Ground( 'GND' , data = GND_data )
F1 = sus.GASFilter( 'F1' , data = F1_data )
CB  = sus.RigidBody( 'CB' , data = CB_data )

F1: the attachment point at the lower end of the internal spring is called 'for_CB'.


In [14]:
# Declare link instances
wire_F1 = sus.Wire( data = wireF1_data )
wire_CB = sus.Wire( data = wireCB_data )

In [15]:
# Connect ground and cube with spring
TS.connect( GND , 'for_F1' , F1 , 'for_GND' , wire_F1 )
TS.connect( F1 , 'for_CB' , CB , 'for_F1' , wire_CB )
GND.show()

GND
└── F1
    └── CB


In [16]:
# print( F1.spring_body )

In [17]:
TS.update()

F1: beta, m1, m2 = -0.001 , 0.1 , 0.0360453484494832


In [18]:
F1.link_internal

In [19]:
info.show_info( TS , 'load')

╭────────┬─────────────┬─────────────┬────────────────┬───────────────╮
│ Name   │   Mass [kg] │   Load [kg] │      Number of │   Single link │
│        │             │             │   links in set │     load [kg] │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ GND    │             │       93.31 │                │               │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ F1     │       91.31 │        2    │              1 │         93.31 │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ CB     │        2    │        0    │              1 │          2    │
╰────────┴─────────────┴─────────────┴────────────────┴───────────────╯


In [20]:
info.show_info( TS , 'links')

╭────────┬──────────────┬────────────────┬───────────────┬─────────────┬─────────────╮
│ Name   │     Link set │      Number of │   Single link │       Axial │   Torsional │
│        │   total load │   links in set │     load [kg] │   stiffness │   stiffness │
│        │         [kg] │                │               │       [N/m] │    [Nm/rad] │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ GND    │              │                │               │             │             │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ F1     │        93.31 │              1 │         93.31 │ 2.21643e+06 │      5353.1 │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ CB     │         2    │              1 │          2    │ 1.68998e+06 │     24408   │
╰────────┴──────────────┴────────────────┴───────────────┴─────────────┴─────────────╯


In [21]:
print( TS.float_coords_list )

None


In [22]:
print( TS.equi_coords_guess_dict )

None


In [23]:
TS.calculate_equilibrium_positions()

Let's calculate the equilibrium positions...
Guessing the values of the equilibrium coordinates... Done.
Calculating the total potential energy...
	Calculating the total wire axial elastic potential energy... Done.
	I'm reasonably assuming that the link elastic torsional potential energies are zero at equilibrium.
Calculating the gradient...
For this task: decimal_places = 20.
Minimizing the total potential energy... Done.
Equilibrium positions found.
With knowledge of the equilibrium positions, I'm calling for the calculation of the link torsional potential energies... 
Calculating the total link torsional elastic potential energy... Done.
Elapsed time: 0.2668 seconds.
Call TS.print_equilibrium_positions() to see the results.


In [24]:
info.print_equilibrium_positions(TS)

╭──────────┬─────────────┬──────────────╮   ╭──────────┬─────────────┬──────────────╮   ╭──────────┬─────────────┬──────────────╮
│ Symbol   │     Guessed │   Calculated │   │ Symbol   │     Guessed │   Calculated │   │ Symbol   │     Guessed │   Calculated │
│          │   value [m] │    value [m] │   │          │   value [m] │    value [m] │   │          │   value [m] │    value [m] │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤
│ lonCB    │       0     │            0 │   │ traCB    │     0       │      0       │   │ rollGND  │           0 │            0 │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤
│ lonF1    │       0     │            0 │   │ traF1    │     0       │      0       │   │ pitchGND │           0 │            0 │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├─

In [25]:
print( TS.float_coords_list )

None


In [26]:
TS.calculate_mass_matrix()

Let's calculate the mass matrix...
Compiling position and velocity coordinates...Done.
	Number of body coordinates: 13
	Number of float coordinates: 0
	Number of ground coordinates: 6
calculating matrix... Done.
Elapsed time: 0.0062 seconds.
Done, call TS.M_matrix to see the result.


In [27]:
print( TS.float_coords_list )

[]


In [28]:
TS.calculate_stiffness_matrix( with_damping = False )

Let's calculate the master stiffness matrix and its submatrices...
The setting `with_damping = False` is suitable for later calculating eigenvectors,
for calculating transfer functions set `with_damping = True`.
For F1:
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
For CB:
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
I have the final master stiffness matrix, you can see it by calling TS.P
Extracting the relevant submatrices from the master stiffness matrix...
	There are no float coordinates, matrices Q, C_XQ, C_QS and related don't exist.
Finished, elapsed time: 3.3550 seconds.


In [29]:
print( is_symmetric( TS.M_matrix ) )
print( is_symmetric( TS.P ) )

True
True


In [30]:
TS.calculate_eigenvectors( TS.M_matrix )

Number of modes: 13
Finished, elapsed time: 0.0110 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


In [31]:
TS.calculate_eigenvectors( TS.P )

Number of modes: 19
Imaginary frequency detected: 1, (0.0 + 7.6879e-23j).
Imaginary frequency detected: 2, (0.0 + 7.1176e-24j).
Imaginary frequency detected: 3, (0.0 + 7.1176e-24j).
Imaginary frequency detected: 4, (0.0 + 1.5352e-76j).
Finished, elapsed time: 0.0946 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


In [32]:
TS.calculate_eigenvectors()

Number of modes: 13
Finished, elapsed time: 0.0521 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


In [33]:
info.print_eigenfreq( TS )

╭──────────┬─────────────╮   ╭──────────┬─────────────╮   ╭──────────┬─────────────╮
│     Mode │   Frequency │   │     Mode │   Frequency │   │     Mode │   Frequency │
│   number │        [Hz] │   │   number │        [Hz] │   │   number │        [Hz] │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        1 │   0.0803166 │   │        6 │    0.672654 │   │       11 │      24.782 │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        2 │   0.0804004 │   │        7 │    0.672654 │   │       12 │    1131.39  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        3 │   0.249999  │   │        8 │    2.2354   │   │       13 │    1435.61  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ╰──────────┴─────────────╯
│        4 │   0.428005  │   │        9 │    2.2354   │
├──────────┼─────────────┤   ├──────────┼─────────────┤
│        5 │   0.428005  │   │       1

In [34]:
n = 2
info.print_eigen( TS , n )

Mode number: 2 

Frequency: 0.08 Hz

╭──────────────┬─────────────╮   ╭──────────────┬────────────╮   ╭──────────────┬──────────────╮
│ Coordinate   │       Value │   │ Coordinate   │      Value │   │ Coordinate   │        Value │
├──────────────┼─────────────┤   ├──────────────┼────────────┤   ├──────────────┼──────────────┤
│ lonF1        │  0.00103679 │   │ yawF1        │ 0          │   │ rollCB       │  0           │
├──────────────┼─────────────┤   ├──────────────┼────────────┤   ├──────────────┼──────────────┤
│ traF1        │  0          │   │ qF1          │ 0          │   │ pitchCB      │ -5.38706e-05 │
├──────────────┼─────────────┤   ├──────────────┼────────────┤   ├──────────────┼──────────────┤
│ verF1        │  0          │   │ lonCB        │ 0.00206743 │   │ yawCB        │  0           │
├──────────────┼─────────────┤   ├──────────────┼────────────┤   ╰──────────────┴──────────────╯
│ rollF1       │  0          │   │ traCB        │ 0          │
├──────────────┼───────────

In [35]:
assert False, "Stopped execution here"

AssertionError: Stopped execution here

In [ ]:
anime.prepare_anime( TS )

In [ ]:
n = 7
# info.print_eigen( TS , n )
# anime.show_anime( TS , n )

In [ ]:
print( F1.spring_load )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TS.calculate_stiffness_matrix( with_damping = True )

In [ ]:
TS.prepare_transfer_functions()

In [ ]:
frequency = np.logspace(-2,4,1000)

In [ ]:
TS.body_coords_list

In [ ]:
in_coord = GND.ver
out_coord = CB.ver
tf = TS.transfer_function( in_coord , out_coord )

In [ ]:
plt.loglog( frequency , np.abs( tf( 2*np.pi*frequency ) ) )
plt.axvline(x=0.25, color='r', linestyle='--', linewidth=2)
plt.axhline(y=1e-3, color='r', linestyle='--')

In [ ]:
7771980404306739/274877906944

In [ ]:
import symengine

# Create a 0x1 dense matrix (vector)
empty_vec = symengine.Matrix(1, 0, []).T

# Convert to list
result = empty_vec.T.tolist()

print(result)  # Output: []
print(type(result))  # Output: <class 'list'>


In [ ]:
import symengine
from symengine import DenseMatrix

# Create a 0x1 empty column vector
v = DenseMatrix(1, 0, []).T 

# Transpose it to a 1x0 row vector
v_T = v.T

# Convert to list: results in a nested empty list [[]]
result = v_T.tolist()

print(f"Original: {v.tolist()}") # Output: []
print(f"Transposed: {result}")   # Output: [[]]
print(f"Shape: {v_T.shape}")      # Output: (1, 0)


In [ ]:
import symengine as se

In [ ]:
isinstance( utils.con( 5 , 5 , 3 ) , se.Number )

In [ ]:
utils.con( 30 , 43 , 6 )

In [ ]:
F1.omega

In [ ]:
F1.omega[2]

In [ ]:
my_keys = GND.gnd_coords
my_values = [ utils.con(0) ] * len( my_keys )
dict( zip( my_keys , my_values ) ) 

In [ ]:
aa = [1.23 , 6.43]

In [ ]:
bb = [aa] if not isinstance( aa , ( list , tuple ) ) else aa

In [ ]:
bb

In [ ]:
utils.con( 1.23 , 6.43 )

In [ ]:
utils.con( 1.23 )